# Study 2: notebook-first reward experiment

**Question:** How do local, global, and mixed rewards change the behavior of four independent Q-learning agents?

The training loop, Q-table updates, and held-out evaluation are visible below. This is a short preview run with one training seed, not the full research comparison.

The current environment uses nonlinear emissions (`8 + 0.10 × EDI + 0.001 × EDI²`), prevention-only mitigation, and high mitigation capacity 20. Economic reward combines EDI gain with `(income - base income) / 10`. Existing displayed outputs may reflect older settings; rerun this notebook to refresh them.

## Settings

In [1]:
import random
import pandas as pd

from study_2_economic_sustainability.agents import TabularQLearner
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment

In [2]:
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEED = 7
TRAINING_EPISODES = 50
EVALUATION_SEEDS = (200_000, 200_001, 200_002, 200_003, 200_004)

## Run one episode

During training, each agent chooses exploratory actions and updates its own Q-table after every step. During evaluation, agents choose greedy actions and Q-values are not updated. The same outcome measures are used for all reward modes.

In [ ]:
def record_step_totals(totals, actions, info):
    totals["gpi"] += info["global_pollution_index"]
    totals["invalid_attempts"] += sum(info["invalid_attempts"])

    for region in range(4):
        if actions[region] in (2, 3, 4):
            totals["mitigation_spending"] += info["costs_paid"][region]


In [ ]:
def make_episode_summary(environment, seed, totals, failed):
    return {
        "seed": seed,
        "steps": environment.step_count,
        "failure": failed,
        "mean_gpi": totals["gpi"] / environment.step_count,
        "final_mean_edi": sum(environment.development_indices) / 4,
        "mitigation_spending": totals["mitigation_spending"],
        "invalid_attempts": totals["invalid_attempts"],
    }


In [ ]:
def run_episode(environment, agents, seed, training):
    observations = environment.reset(seed)
    totals = {"gpi": 0.0, "mitigation_spending": 0.0, "invalid_attempts": 0}

    while True:
        actions = []
        for region in range(4):
            action = agents[region].select_action(
                observations[region], greedy=not training
            )
            actions.append(action)

        next_observations, rewards, terminated, truncated, info = environment.step(
            tuple(actions)
        )

        if training:
            for region in range(4):
                agents[region].update(
                    observations[region], actions[region], rewards[region],
                    next_observations[region], terminated,
                )

        record_step_totals(totals, actions, info)
        observations = next_observations

        if terminated or truncated:
            break

    return make_episode_summary(environment, seed, totals, terminated)


## Experiment 1: compare reward modes

Each reward mode gets four fresh agents. They use the same training hyperparameters and training episode seed sequence. The held-out seeds are outside the training seed range.

In [4]:
evaluation_rows = []
learned_agents = {}

for reward_mode in REWARD_MODES:
    environment = EconomicSustainabilityEnvironment(reward_mode)
    episode_rng = random.Random(TRAINING_SEED)
    agents = []

    for region in range(4):
        agents.append(TabularQLearner(TRAINING_SEED * 10 + region))

    for episode in range(TRAINING_EPISODES):
        training_environment_seed = episode_rng.randrange(100_000)
        run_episode(environment, agents, training_environment_seed, training=True)

        for agent in agents:
            agent.decay_epsilon()

    learned_agents[reward_mode] = agents

    for evaluation_seed in EVALUATION_SEEDS:
        row = run_episode(environment, agents, evaluation_seed, training=False)
        row["reward_mode"] = reward_mode
        evaluation_rows.append(row)

evaluations = pd.DataFrame(evaluation_rows)
display(evaluations)

,seed,steps,failure,mean_gpi,final_mean_edi,mitigation_spending,invalid_attempts,reward_mode
0,200000,200,False,7.277837,94.5,31155.0,39,local
1,200001,200,False,4.792903,92.5,34200.0,19,local
2,200002,200,False,7.916858,92.5,32920.0,27,local
3,200003,200,False,4.951687,92.5,33910.0,19,local
4,200004,200,False,5.602290,92.5,32610.0,30,local
5,200000,200,False,18.942909,23.0,13785.0,4,global
6,200001,200,False,12.625534,21.5,15585.0,5,global
7,200002,200,False,9.695020,0.0,14125.0,37,global
8,200003,200,False,35.109984,21.0,8405.0,5,global
9,200004,200,False,36.993761,56.5,13500.0,14,global


- `steps` is the episode length
- `failure` tells us whether pollution reached the failure threshold
- `mean_gpi` is the Global Pollution Index averaged over the episode and lower values mean less pollution
- `final_mean_edi` is the average final development index of the four regions, note that higher values mean more development
- `mitigation_spending` adds the four regions' mitigation costs over the episode
 - `invalid_attempts` counts actions the environment rejected.

## Compare the outcomes

Training rewards have different definitions, so compare pollution, development, spending, and failures instead of ranking reward modes by reward magnitude.

In [5]:
summary = evaluations.groupby("reward_mode", sort=False).agg(
    mean_gpi=("mean_gpi", "mean"),
    final_mean_edi=("final_mean_edi", "mean"),
    mitigation_spending=("mitigation_spending", "mean"),
    failure_rate=("failure", "mean"),
    invalid_attempts=("invalid_attempts", "sum"),
)

display(summary.round(2))

,mean_gpi,final_mean_edi,mitigation_spending,failure_rate,invalid_attempts
reward_mode,,,,,
local,6.11,92.9,32959.0,0.0,134
global,22.67,24.4,13080.0,0.0,65
mixed,7.42,71.6,31505.0,0.0,128


### Initial comparison

- **local** had the lowest mean GPI (6.11) and highest final EDI (92.9), but also the highest mitigation spending (32,959 per episode)
- **Mixed** was close in GPI (7.42), with lower final EDI (71.6) and slightly lower spending (31,505)
- **Global** spent much less (13,080), but had higher mean GPI (22.67) and much lower final EDI (24.4).

## Inspect the Q-tables

In [6]:
q_table_sizes = []

for reward_mode, agents in learned_agents.items():
    for region, agent in enumerate(agents):
        q_table_sizes.append({
            "reward_mode": reward_mode,
            "region": region,
            "observations_in_q_table": len(agent.q),
            "epsilon_after_training": agent.epsilon,
        })

display(pd.DataFrame(q_table_sizes))

,reward_mode,region,observations_in_q_table,epsilon_after_training
0,local,0,44,0.778313
1,local,1,41,0.778313
2,local,2,44,0.778313
3,local,3,39,0.778313
4,global,0,44,0.778313
5,global,1,45,0.778313
6,global,2,43,0.778313
7,global,3,43,0.778313
8,mixed,0,44,0.778313
9,mixed,1,43,0.778313


### What the Q-table inspection tells us

Epsilon is still about 0.78 after 50 training episodes, so exploration remains high at the end of this preview run.

Note that evaluation uses greedy actions. We would need longer training and multiple independent training seeds before drawing a stronger conclusion from the reward comparison.